In [1]:
import re
import pandas as pd
from collections import Counter
from datetime import datetime
import numpy as np

# -----------------------------
# STEP 1: LOAD & PARSE CHAT
# -----------------------------
file_path = "mabel.txt"

pattern = r"(\d{2}/\d{2}/\d{2},\s\d{1,2}:\d{2}\s(?:am|pm))\s-\s([^:]+):\s(.*)"

data = []

with open(file_path, encoding='utf-8') as f:
    for line in f:
        match = re.match(pattern, line)
        if match:
            date, user, message = match.groups()
            data.append([date, user.strip(), message.strip()])

df = pd.DataFrame(data, columns=["datetime", "user", "message"])

# Convert datetime
df["datetime"] = pd.to_datetime(df["datetime"], format="%d/%m/%y, %I:%M %p")

# Extract time features
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day_name()

# -----------------------------
# STEP 2: BASIC COUNTS
# -----------------------------
total_msgs = len(df)
msgs_per_user = df["user"].value_counts()

# Avg messages per day
avg_msgs_per_day = df.groupby("date").size().mean()

# Most active
most_active_date = df["date"].value_counts().idxmax()
most_active_day = df["day"].value_counts().idxmax()
most_active_hour = df["hour"].value_counts().idxmax()

# -----------------------------
# STEP 3: MEDIA / DELETED / CALLS
# -----------------------------
media_count = df[df["message"].str.contains("file attached|<Media omitted>", na=False)].shape[0]

deleted_count = df[df["message"].str.contains("deleted this message", case=False, na=False)].shape[0]

missed_voice_calls = df[df["message"].str.contains("missed voice call", case=False, na=False)].shape[0]
missed_video_calls = df[df["message"].str.contains("missed video call", case=False, na=False)].shape[0]

# -----------------------------
# STEP 4: TALKATIVE SCORE
# -----------------------------
user_stats = df.groupby("user").agg({
    "message": ["count", lambda x: np.mean([len(i) for i in x])]
})

user_stats.columns = ["msg_count", "avg_msg_length"]

# Normalize
user_stats["talk_score"] = (
    user_stats["msg_count"] / user_stats["msg_count"].sum() +
    user_stats["avg_msg_length"] / user_stats["avg_msg_length"].sum()
)

# Classification
def classify_talk(score):
    if score > user_stats["talk_score"].mean():
        return "Talkative"
    else:
        return "Less Talkative"

user_stats["talk_type"] = user_stats["talk_score"].apply(classify_talk)

# -----------------------------
# STEP 5: FLIRT DETECTION (NLP)
# -----------------------------
flirt_keywords = [
    "love", "dear", "miss you", "❤️", "😍", "😘", "baby", "sweet",
    "darling", "cute", "😘", "🥰"
]

def flirt_score(msg):
    msg_lower = msg.lower()
    return sum(word in msg_lower for word in flirt_keywords)

df["flirt_score"] = df["message"].apply(flirt_score)

flirt_per_user = df.groupby("user")["flirt_score"].sum()

# Normalize flirt %
flirt_percent = (flirt_per_user / flirt_per_user.sum()) * 100

# -----------------------------
# STEP 6: FINAL REPORT
# -----------------------------
print("\n📊 OVERALL STATS")
print("Total Messages:", total_msgs)
print("Average Messages per Day:", round(avg_msgs_per_day, 2))
print("Most Active Date:", most_active_date)
print("Most Active Day:", most_active_day)
print("Most Active Hour:", most_active_hour)

print("\n📁 MEDIA & CALL STATS")
print("Media Count:", media_count)
print("Deleted Messages:", deleted_count)
print("Missed Voice Calls:", missed_voice_calls)
print("Missed Video Calls:", missed_video_calls)

print("\n👥 USER ANALYSIS")
for user in user_stats.index:
    print(f"\nUser: {user}")
    print("Message Count:", user_stats.loc[user, "msg_count"])
    print("Talk Type:", user_stats.loc[user, "talk_type"])
    print("Talk %:", round(user_stats.loc[user, "talk_score"] * 100, 2))
    print("Flirt %:", round(flirt_percent.get(user, 0), 2))


📊 OVERALL STATS
Total Messages: 52
Average Messages per Day: 7.43
Most Active Date: 2019-12-05
Most Active Day: Thursday
Most Active Hour: 13

📁 MEDIA & CALL STATS
Media Count: 1
Deleted Messages: 1
Missed Voice Calls: 0
Missed Video Calls: 0

👥 USER ANALYSIS

User: AR❤
Message Count: 29
Talk Type: Talkative
Talk %: 103.98
Flirt %: nan

User: Mabel Infoziant
Message Count: 23
Talk Type: Less Talkative
Talk %: 96.02
Flirt %: nan
